In [ ]:
import allel
import numpy as np
import dask.array as da
import xarray
import pandas as pd
import gcsfs
from pyprojroot import here
import logging
from tqdm import tqdm
from collections import Counter
# While we are not connected to the API yet, there is a lot of functionality here that we can use
from malariagen_data import util

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
%run {here()}/setup.ipynb
gcs = init_gcs()

In [ ]:
# Set up Dask cluster
from dask_gateway import Gateway
gateway = Gateway()

conda_prefix = os.environ['CONDA_PREFIX']
current_environment = re.sub("/home/conda/(.+)/envs/(.+)","\\1/\\2", conda_prefix)
# Create cluster
cluster = gateway.new_cluster(
        profile="highmem",
        conda_environment = current_environment,
)
cluster.scale(60)

client = cluster.get_client()
client

In [ ]:
# Define helpful paths and helper vars

# Get all contigs in the genome
genome_path = "gs://vo_adir_release/reference/genome/AdirusWRAIR2/VectorBase-56_AdirusWRAIR2_Genome.zarr"
genome_store = gcs.get_mapper(genome_path)
genome = zarr.open_consolidated(genome_store)

contig_lengths = {
    k: len(genome[k]) for k in genome
}

contigs = sorted(contig_lengths, key=contig_lengths.get, reverse=True)[:79] # Filter down to called contigs

# Define sample sets
sample_sets = [
    "1278-VO-TH-KOBYLINSKI-VMF00153",
    "1276-AD-BD-ALAM-VMF00156",
    "1277-VO-KH-WITKOWSKI-VMF00151",
    "1277-VO-KH-WITKOWSKI-VMF00183",
]

# Paths to SNP genotype data
base_bucket_path = "gs://vo_adir_production_us_central1/v1.x/"
staging_bucket = base_bucket_path + 'staging/'
staged_snps_pattern = "gs://vo_adir_production_us_central1/v1.x/staging/snp_genotypes/all/{sample_set}"
gcs_release_manifest_path = f'{staging_bucket}/manifest.tsv'

# Path to POS data
variant_zarr_url = "gs://vo_adir_release/reference/genome/AdirusWRAIR2/VectorBase-56_AdirusWRAIR2_Genome_allsites_nonN.zarr"

# Path to filter mask data
qc_stat_zarr_url = 'gs://vo_adir_production_us_central1/accessibility_summary_stats/v1.0/snp_accessibility_filtering.zarr'

In [ ]:
# Load variant_position
def open_variant_array(v, contig):
    root = open_gcs_zarr(gcs_url=variant_zarr_url, gcs=gcs)
    array = root[f"{contig}/variants/{v}"][:]  # Load entire array into memory

    qc_store = gcs.get_mapper(qc_stat_zarr_url)
    qc_data = zarr.open(qc_store)
    mask = qc_data[f"{contig}/sc_filter_pass"][:]  # Also fully in memory

    return np.compress(mask, array, axis=0)

def open_calldata_array(v, contig, sample_sets):

    if isinstance(sample_sets, str):
        sample_set = [sample_sets]
        
    arrays = []
    
    for sample_set in sample_sets:
        url = staged_snps_pattern.format(sample_set=sample_set)
        root = open_gcs_zarr(gcs_url=url, gcs=gcs)
        array = root[f"{contig}/calldata/{v}"]
        arrays.append(array)

    return da.concatenate(arrays, axis=1)


def load_metadata():
    dfs = []
    for sample_set in sample_sets:
        d = pd.read_csv(metadata_url.format(sample_set = sample_set))
        dfs.append(d)
    return pd.concat(dfs)

# Get genotypes from the longest contig, subset to samples passing QC and passing filters
def load_gts(contig, sample_set, max_cohort_size=None):
    url = staged_snps_pattern.format(sample_set=sample_set)
    root = open_gcs_zarr(gcs_url=url, gcs=gcs)
    array = allel.GenotypeArray(root[f"{contig}/calldata/GT"])
    
    # Apply filter mask
    qc_store = gcs.get_mapper(qc_stat_zarr_url)
    qc_data = zarr.open(qc_store)
    array = array.compress(qc_data[f"{contig}/sc_filter_pass"][:], axis=0)
    
    # Subset to max cohort size if asked
    if max_cohort_size < array.shape[1]:
        array = array[:,np.random.choice(array.shape[1], max_cohort_size, replace=False)]
    else:
        pass
    
    return array
   
def diplotype_frequencies(gt):
    """Compute diplotype frequencies, returning a dictionary that maps
    diplotype hash values to frequencies."""

    # Here are some optimisations to speed up the computation
    # of diplotype hashes. First we combine the two int8 alleles
    # in each genotype call into a single int16.
    m = gt.shape[0]
    n = gt.shape[1]
    x = np.asarray(gt).view(np.int16).reshape((m, n))

    # Now call optimised hashing function.
    hashes = util.hash_columns(x)

    # Now compute counts and frequencies of distinct haplotypes.
    counts = Counter(hashes)
    freqs = {key: count / n for key, count in counts.items()}

    return freqs


def garud_g123(gt):
    """Compute Garud's G123."""

    # compute diplotype frequencies
    frq_counter = diplotype_frequencies(gt)

    # convert to array of sorted frequencies
    f = np.sort(np.fromiter(frq_counter.values(), dtype=float))[::-1]

    # compute G123
    g123 = np.sum(f[:3]) ** 2 + np.sum(f[3:] ** 2)

    # These other statistics are not currently needed, but leaving here
    # commented out for future reference...

    # compute G1
    # g1 = np.sum(f**2)

    # compute G12
    # g12 = np.sum(f[:2]) ** 2 + np.sum(f[2:] ** 2)  # type: ignore[index]

    # compute G2/G1
    # g2 = g1 - f[0] ** 2  # type: ignore[index]
    # g2_g1 = g2 / g1

    return g123


def g123_gwss(
    gt, # genotypes as 0,1,2 (n_alt)
    pos,
    window_size,
):

    g123 = allel.moving_statistic(gt, statistic=garud_g123, size=window_size)
    x = allel.moving_statistic(pos, statistic=np.mean, size=window_size)

    return x, g123

def plot_g123_gwss(
    gt,
    pos,
    window_size,
    contig = None,
    title=None,
    color='black',
    width=800,
    height=500,
    show=True,
):

    import plotly.express as px

    x, g123 = g123_gwss(gt=gt, pos=pos, window_size=window_size)
    
    g123_df = pd.DataFrame({'pos':x, 'g123':g123, 'contig' : contig})
    
    # fig = px.scatter(g123_df, 
    #                 x='pos', 
    #                 y='g123', 
    #                 template='simple_white',
    #                 color_discrete_sequence=[color], 
    #                 title=title, 
    #                 width=width, 
    #                 height=height)

    return g123_df #fig

In [ ]:

def plot_combined_g123(sample_set, 
                       winsize, 
                       max_cohort_size=None):

    df_contigs = []

    # Progress bar for contigs
    for contig in tqdm(contigs, desc=f"Processing contigs for {sample_set}"):
        
        # Load genos
        b_gt = load_gts(
            contig, 
            sample_set=sample_set, 
            max_cohort_size=max_cohort_size
        )
        
        # Load pos
        pos = open_variant_array('POS', contig)
        
        # Calculate G123 for contig
        b_g = plot_g123_gwss(
            b_gt, pos, winsize, 
            title=sample_set, 
            contig=contig
        )    

        df_contigs.append(b_g)

    # Concatenate df and save out
    df = pd.concat(df_contigs)
    df.to_csv(f'{sample_set}.{winsize}.g123.csv')
    print(f"DataFrame saved as {sample_set}.{winsize}.g123.csv")    
    
    # Sort contigs by length (longest first)
    contig_lengths = df.groupby('contig')['pos'].max()
    sorted_contigs = contig_lengths.sort_values(ascending=False).index.tolist()
    sorted_lengths = contig_lengths.loc[sorted_contigs]
    contig_offsets = sorted_lengths.cumsum().shift(fill_value=0)

    df['contig'] = pd.Categorical(df['contig'], categories=sorted_contigs, ordered=True)
    df = df.sort_values(['contig', 'pos']).copy()
    df['contig_offset'] = df['contig'].map(contig_offsets).astype(float)
    df['genome_position'] = df['pos'] + df['contig_offset']

    color_map = {
        contig: ('lightblue' if i % 2 == 0 else 'steelblue')
        for i, contig in enumerate(sorted_contigs)
    }
    df['color'] = df['contig'].map(color_map)

    fig = px.scatter(
        df,
        x='genome_position',
        y='g123',
        color='contig',
        color_discrete_map=color_map,
        labels={'genome_position': 'Genomic Position', 'g123': 'G123'},
        title=f'{sample_set}'
    )

    fig.update_traces(marker=dict(size=3))  
    fig.write_image(f"figures/{sample_set}.{winsize}.g123.png")
    print(f"Figure saved as figures/{sample_set}.{winsize}.g123.png")


In [ ]:
for sample_set in sample_sets:
    #print(sample_set)
    plot_combined_g123(sample_set = sample_set, winsize=2000, max_cohort_size=50)

In [ ]:
# Once this is plugged in to the API we can use the caching easily to make this slicker
dflist = []
for s in sample_sets:
    t = pd.read_csv(f"{s}.2000.g123.csv")
    dflist.append(t)

In [ ]:
def plot_gwss(winsize, sample_set):
    
    df = pd.read_csv(f"{sample_set}.{winsize}.g123.csv")
    
    # Sort contigs by length (longest first)
    contig_lengths = df.groupby('contig')['pos'].max()
    sorted_contigs = contig_lengths.sort_values(ascending=False).index.tolist()
    sorted_lengths = contig_lengths.loc[sorted_contigs]
    contig_offsets = sorted_lengths.cumsum().shift(fill_value=0)

    df['contig'] = pd.Categorical(df['contig'], categories=sorted_contigs, ordered=True)
    df = df.sort_values(['contig', 'pos']).copy()
    df['contig_offset'] = df['contig'].map(contig_offsets).astype(float)
    df['genome_position'] = df['pos'] + df['contig_offset']

    color_map = {
        contig: ('lightblue' if i % 2 == 0 else 'steelblue')
        for i, contig in enumerate(sorted_contigs)
    }
    df['color'] = df['contig'].map(color_map)

    fig = px.scatter(
        df,
        x='genome_position',
        y='g123',
        color='contig',
        color_discrete_map=color_map,
        template='simple_white',
        labels={'genome_position': 'Genomic Position', 'g123': 'G123'},
        title=f'{sample_set}',
        hover_data=["contig", "pos"]
    )
    
    fig.update_layout(showlegend=False) #Hide legend
    fig.update_traces(marker=dict(size=3))  
    fig.show()

    #fig.write_image(f"figures/{sample_set}.{winsize}.g123.png")
    #print(f"Figure saved as figures/{sample_set}.{winsize}.g123.png")

In [ ]:
for sample_set in sample_sets:
    plot_gwss(winsize=2000, sample_set=sample_set)